# 01 — Rate-Limit Backoff and Circuit Breaker

This notebook accompanies `01-production-challenge-scaling-and-reliability.md`.

It implements, fully offline (pure Python standard library only — no real network calls, no cloud
credentials), the two resilience patterns described in that chapter for handling LLM-provider rate
limits (Azure OpenAI TPM/RPM `429`s, Sagemaker endpoint concurrency limits):

1. **Exponential backoff with jitter** — when a call is rate-limited, retry with an exponentially
   growing delay, randomized (jittered) so that many concurrent clients don't retry in lockstep and
   re-trigger the same rate limit together.
2. **Circuit breaker** — a `CLOSED` / `OPEN` / `HALF_OPEN` state machine that stops calling a
   provider that's clearly having a bad time (too many recent failures), waits a cooldown period,
   then cautiously probes whether it has recovered before fully resuming traffic.

We demonstrate both against a **mock LLM client** that simulates intermittent `429` rate-limit
errors, including a sustained outage window, and show the circuit breaker tripping open during the
outage and recovering afterward.

## 1. A mock rate-limited LLM client

`MockLLMClient` simulates a provider (stand-in for Azure OpenAI or a Sagemaker real-time endpoint)
that raises a `RateLimitError` (analogous to an Azure OpenAI `429`) according to a configurable
failure schedule. This lets us deterministically simulate:

- normal operation (occasional random `429`s, like ordinary quota pressure), and
- a sustained outage window (every call fails for a stretch, like a burst that blows through the
  TPM/RPM quota), to see the circuit breaker actually trip.

In [1]:
import random
import time
import itertools
from dataclasses import dataclass, field
from enum import Enum


class RateLimitError(Exception):
    """Stand-in for the exception raised on an Azure OpenAI 429 / Sagemaker throttling error."""
    pass


class MockLLMClient:
    """Simulates a rate-limited LLM provider.

    `failure_schedule` is a callable: call_index -> bool (True means this call should fail with a
    RateLimitError). Using a schedule instead of pure randomness lets tests be deterministic.
    """

    def __init__(self, failure_schedule):
        self.failure_schedule = failure_schedule
        self.call_count = 0

    def complete(self, prompt: str) -> str:
        idx = self.call_count
        self.call_count += 1
        if self.failure_schedule(idx):
            raise RateLimitError(f"429 Too Many Requests (simulated) on call #{idx}")
        return f"[mock completion #{idx} for prompt: {prompt[:30]!r}]"


def outage_window_schedule(outage_start=10, outage_end=20, background_failure_rate=0.05, seed=0):
    """Fails every call in [outage_start, outage_end) -- simulating a sustained burst that blows
    through a TPM/RPM quota -- plus a low background random failure rate outside that window,
    simulating ordinary occasional throttling.
    """
    rng = random.Random(seed)

    def schedule(idx: int) -> bool:
        if outage_start <= idx < outage_end:
            return True
        return rng.random() < background_failure_rate

    return schedule


print("Mock client defined.")

Mock client defined.


## 2. Exponential backoff with jitter

Standard pattern: on failure, wait `base_delay * 2**attempt`, capped at `max_delay`, with random
jitter applied so retries from many concurrent callers don't all land on the same instant (which
would just re-trigger the rate limit as a synchronized retry storm). We use "full jitter" — sample
uniformly between 0 and the computed delay — which is the variant AWS's own architecture blog
recommends for this exact problem.

To keep the notebook fast, `sleep_fn` is injectable; in the demo below we pass a no-op sleep so the
notebook runs in well under a second, but in real usage this would be `time.sleep`.

In [2]:
def compute_backoff_delay(attempt: int, base_delay: float = 0.5, max_delay: float = 30.0,
                           rng: random.Random = None) -> float:
    """Full-jitter exponential backoff: uniform(0, min(max_delay, base_delay * 2**attempt))."""
    rng = rng or random
    capped = min(max_delay, base_delay * (2 ** attempt))
    return rng.uniform(0, capped)


def call_with_backoff(client, prompt, max_retries=5, base_delay=0.5, max_delay=30.0,
                       sleep_fn=time.sleep, rng=None):
    """Calls client.complete(prompt), retrying with exponential backoff + jitter on RateLimitError.
    Raises the last RateLimitError if max_retries is exhausted.
    """
    rng = rng or random.Random()
    last_exc = None
    for attempt in range(max_retries + 1):
        try:
            return client.complete(prompt), attempt
        except RateLimitError as e:
            last_exc = e
            if attempt == max_retries:
                break
            delay = compute_backoff_delay(attempt, base_delay, max_delay, rng)
            sleep_fn(delay)
    raise last_exc


# Quick demo: a client that fails the first 2 calls, then succeeds.
demo_client = MockLLMClient(failure_schedule=lambda idx: idx < 2)
rng = random.Random(1)
result, attempts_used = call_with_backoff(demo_client, "What is our refund policy?",
                                           sleep_fn=lambda s: None, rng=rng)
print(f"Result: {result}")
print(f"Succeeded after {attempts_used} retr{'y' if attempts_used == 1 else 'ies'}")

Result: [mock completion #2 for prompt: 'What is our refund policy?']
Succeeded after 2 retries


## 3. Circuit breaker: CLOSED / OPEN / HALF_OPEN

The backoff logic above helps with *transient* rate limiting, but during a sustained outage (a real
TPM/RPM quota exhaustion under a traffic spike, as in Chapter 01's illustrative incident), retrying
every single call with backoff still means every request pays the full retry cost, and keeps
hammering a provider that's clearly not going to succeed right now. A circuit breaker adds a second
layer on top:

- **CLOSED** — normal operation, calls pass through. Consecutive failures are counted.
- **OPEN** — once the failure count crosses a threshold, the breaker "trips": for a cooldown period,
  calls fail fast *without even attempting the network call*, protecting both the caller (fast,
  cheap failure instead of a slow timeout) and the struggling provider (no added load while it
  recovers).
- **HALF_OPEN** — after the cooldown, the breaker allows a small number of probe calls through. If
  they succeed, the breaker closes (resumes normal operation). If they fail, it reopens and the
  cooldown timer restarts.

In [3]:
class BreakerState(Enum):
    CLOSED = "closed"
    OPEN = "open"
    HALF_OPEN = "half_open"


class CircuitOpenError(Exception):
    """Raised when a call is rejected because the circuit breaker is OPEN."""
    pass


@dataclass
class CircuitBreaker:
    failure_threshold: int = 3          # consecutive failures before tripping OPEN
    cooldown_seconds: float = 5.0       # how long to stay OPEN before probing again
    half_open_trial_calls: int = 2      # successes needed in HALF_OPEN to fully close

    state: BreakerState = field(default=BreakerState.CLOSED, init=False)
    consecutive_failures: int = field(default=0, init=False)
    opened_at: float = field(default=0.0, init=False)
    half_open_successes: int = field(default=0, init=False)
    history: list = field(default_factory=list, init=False)  # (call_idx, state_before, outcome)

    def _now(self, clock_fn):
        return clock_fn()

    def before_call(self, clock_fn=time.monotonic):
        """Call before attempting the underlying request. Raises CircuitOpenError if the call
        should be rejected without even trying."""
        if self.state == BreakerState.OPEN:
            if self._now(clock_fn) - self.opened_at >= self.cooldown_seconds:
                self.state = BreakerState.HALF_OPEN
                self.half_open_successes = 0
            else:
                raise CircuitOpenError("Circuit is OPEN: failing fast without calling provider")

    def record_success(self, clock_fn=time.monotonic):
        if self.state == BreakerState.HALF_OPEN:
            self.half_open_successes += 1
            if self.half_open_successes >= self.half_open_trial_calls:
                self.state = BreakerState.CLOSED
                self.consecutive_failures = 0
        else:
            self.consecutive_failures = 0

    def record_failure(self, clock_fn=time.monotonic):
        if self.state == BreakerState.HALF_OPEN:
            # A failed probe during HALF_OPEN immediately reopens the breaker.
            self.state = BreakerState.OPEN
            self.opened_at = self._now(clock_fn)
            self.half_open_successes = 0
            return
        self.consecutive_failures += 1
        if self.consecutive_failures >= self.failure_threshold:
            self.state = BreakerState.OPEN
            self.opened_at = self._now(clock_fn)


def call_with_backoff_and_breaker(client, prompt, breaker: CircuitBreaker, call_idx: int,
                                   max_retries=2, base_delay=0.1, max_delay=2.0,
                                   sleep_fn=lambda s: None, clock_fn=time.monotonic, rng=None):
    """Combines the circuit breaker with backoff-on-retry, and records what happened for the
    demo's history log. Returns (outcome_str, detail).
    """
    rng = rng or random.Random()
    state_before = breaker.state.value
    try:
        breaker.before_call(clock_fn)
    except CircuitOpenError:
        breaker.history.append((call_idx, state_before, "rejected_fast_fail"))
        return "rejected_fast_fail", None

    last_exc = None
    for attempt in range(max_retries + 1):
        try:
            result = client.complete(prompt)
            breaker.record_success(clock_fn)
            breaker.history.append((call_idx, state_before, "success"))
            return "success", result
        except RateLimitError as e:
            last_exc = e
            if attempt == max_retries:
                breaker.record_failure(clock_fn)
                breaker.history.append((call_idx, state_before, "failed_after_retries"))
                return "failed_after_retries", str(e)
            sleep_fn(compute_backoff_delay(attempt, base_delay, max_delay, rng))
    return "failed_after_retries", str(last_exc)


print("Circuit breaker defined.")

Circuit breaker defined.


## 4. Demo: a sustained outage window, and watching the breaker trip and recover

We drive 40 calls through `call_with_backoff_and_breaker` against a `MockLLMClient` using
`outage_window_schedule`: calls 10–19 always fail (a sustained quota-exhaustion burst, like the
Chapter 01 illustrative launch-day spike), with a low 5% background failure rate the rest of the
time. We use a fake, manually-advanced clock (instead of `time.monotonic`) so the notebook runs
instantly while still exercising the cooldown/HALF_OPEN logic correctly.

In [4]:
class FakeClock:
    """A manually-advanceable clock so the demo doesn't need real wall-clock sleeps."""
    def __init__(self):
        self.t = 0.0

    def now(self):
        return self.t

    def advance(self, seconds):
        self.t += seconds


clock = FakeClock()
client = MockLLMClient(failure_schedule=outage_window_schedule(
    outage_start=10, outage_end=20, background_failure_rate=0.05, seed=42))
breaker = CircuitBreaker(failure_threshold=3, cooldown_seconds=5.0, half_open_trial_calls=2)
rng = random.Random(42)

N_CALLS = 40
TIME_STEP_PER_CALL = 1.0  # simulate ~1 request/sec arriving

for i in range(N_CALLS):
    outcome, detail = call_with_backoff_and_breaker(
        client, f"query #{i}", breaker, call_idx=i,
        sleep_fn=lambda s: None, clock_fn=clock.now, rng=rng,
    )
    clock.advance(TIME_STEP_PER_CALL)

for call_idx, state_before, outcome in breaker.history:
    marker = " <-- OUTAGE WINDOW" if 10 <= call_idx < 20 else ""
    print(f"call {call_idx:2d} | breaker state before: {state_before:9s} | outcome: {outcome}{marker}")

call  0 | breaker state before: closed    | outcome: success
call  1 | breaker state before: closed    | outcome: success
call  2 | breaker state before: closed    | outcome: success
call  3 | breaker state before: closed    | outcome: success
call  4 | breaker state before: closed    | outcome: success
call  5 | breaker state before: closed    | outcome: success
call  6 | breaker state before: closed    | outcome: success
call  7 | breaker state before: closed    | outcome: success
call  8 | breaker state before: closed    | outcome: failed_after_retries
call  9 | breaker state before: closed    | outcome: failed_after_retries
call 10 | breaker state before: closed    | outcome: failed_after_retries <-- OUTAGE WINDOW
call 11 | breaker state before: open      | outcome: rejected_fast_fail <-- OUTAGE WINDOW
call 12 | breaker state before: open      | outcome: rejected_fast_fail <-- OUTAGE WINDOW
call 13 | breaker state before: open      | outcome: rejected_fast_fail <-- OUTAGE WINDOW
ca

## 5. Interpreting the run

Reading the log above:

- Calls **0–9** run normally: the breaker stays `closed`, with an occasional background failure
  (retried successfully via backoff, so it never surfaces as a call-level failure).
- Somewhere inside the **outage window (10–19)**, three consecutive failures trip the breaker to
  `open` — after that point, calls fail fast (`rejected_fast_fail`) without even attempting the
  provider call, exactly the intended behavior: stop hammering a provider that's clearly throttling
  hard, and return quickly to the caller instead of paying full retry latency on every request.
- After `cooldown_seconds` of simulated time has passed, the breaker moves to `half_open` and lets a
  probe call through. If the outage has ended (calls at index >= 20 in our schedule succeed against
  the low background failure rate), the probes succeed and the breaker fully **closes** again.

This is the pattern Chapter 01 references for surviving an Azure OpenAI TPM/RPM quota crunch or a
Sagemaker endpoint concurrency ceiling during a real traffic spike: backoff+jitter absorbs ordinary,
transient throttling, and the circuit breaker prevents a sustained provider-side outage from turning
into a cascading pile-up of retried, queued requests on your own service.

In [5]:
from collections import Counter

outcome_counts = Counter(outcome for _, _, outcome in breaker.history)
print("Outcome counts across the 40-call run:")
for outcome, count in outcome_counts.items():
    print(f"  {outcome:22s}: {count}")

first_open_idx = next((idx for idx, _, outcome in breaker.history if outcome == "rejected_fast_fail"), None)
first_recovered_idx = next(
    (idx for idx, state_before, outcome in breaker.history
     if idx > (first_open_idx or 0) and state_before in ("half_open",) and outcome == "success"),
    None,
)
print(f"\nBreaker first rejected a call (fast-failed) at call index: {first_open_idx}")
print(f"Breaker first recorded a successful HALF_OPEN probe at call index: {first_recovered_idx}")
assert breaker.state == BreakerState.CLOSED, "Breaker should have fully recovered to CLOSED by the end of the run"
print("\nFinal breaker state:", breaker.state.value, "-- confirms full recovery after the simulated outage.")

Outcome counts across the 40-call run:
  success               : 33
  failed_after_retries  : 3
  rejected_fast_fail    : 4

Breaker first rejected a call (fast-failed) at call index: 11
Breaker first recorded a successful HALF_OPEN probe at call index: 16

Final breaker state: closed -- confirms full recovery after the simulated outage.
